In [ ]:
!rm -rf /kaggle/working/ML-for-Wireless-Security-Threats-Identification
!git clone https://github.com/duynhat326/ML-for-Wireless-Security-Threats-Identification.git /kaggle/working/ML-for-Wireless-Security-Threats-Identification

In [ ]:
import os, sys, time
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT", "/kaggle/working/ML-for-Wireless-Security-Threats-Identification"))
sys.path.insert(0, str(ROOT))
RESULTS_DIRECTORY, SAVED_MODEL_DIRECTORY = Path("results"), Path("saved_model")
RESULTS_DIRECTORY.mkdir(exist_ok=True)
SAVED_MODEL_DIRECTORY.mkdir(exist_ok=True)
import numpy as np
from shared.config import LABELS, SEED
from shared.data_loader import load_splits
from shared.evaluator import measure_latency_milliseconds, save_json, save_results
from shared.tracer import build_alerts, save_bundle
import lightgbm as lgb

In [ ]:
splits = load_splits()
counts = np.bincount(splits["labels_train"], minlength=len(LABELS))
raw_weights = np.sqrt(counts.max() / np.maximum(counts, 1))
soft_class_weight = {i: float(min(w, 20.0)) for i, w in enumerate(raw_weights)}

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight=soft_class_weight,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)
start_time = time.perf_counter()
model.fit(
    splits["features_train"], splits["labels_train"],
    eval_set=[(splits["features_val"], splits["labels_val"])],
    callbacks=[lgb.early_stopping(30, verbose=True), lgb.log_evaluation(50)],
)
train_time = time.perf_counter() - start_time

In [ ]:
predictions = model.predict(splits["features_test"])
latency = measure_latency_milliseconds(model.predict, splits["features_test"])
metrics = save_results(RESULTS_DIRECTORY, "lightgbm", splits["labels_test"], predictions, train_time, latency,
                       (splits["features_train"].columns, model.feature_importances_))
save_bundle(SAVED_MODEL_DIRECTORY / "model.pkl", model, splits["schema"])

import pandas as pd
print(pd.read_csv("results/per_class_report.csv").to_string(index=False))

In [ ]:
from shared.tracer import ScikitLearnPredictor
probabilities = ScikitLearnPredictor(model, splits["schema"]).predict_proba(splits["features_test"])
alerts = build_alerts(splits["metadata_test"], probabilities.argmax(axis=1), probabilities.max(axis=1))
save_json(alerts, RESULTS_DIRECTORY / "trace_results.json")
print(metrics)
print(len(alerts), "incidents")